# XGBoost -- Leaf-Wise Growth (lossguide) Tuned Model

Your last run (`max_depth=9`, depth-wise growth) improved Top-3/Top-5/F1 but left Top-1 flat (79.87% vs 79.92%) -- the extra depth helped ranking, not the single best guess. This version switches to `grow_policy="lossguide"`, XGBoost's leaf-wise tree growth (same idea LightGBM uses): trees expand wherever loss reduction is highest, not uniformly by depth -- more efficient use of tree capacity on a problem like this (727 classes, purely binary symptom features, a few specific combinations carrying most of the signal). Still a single training pass, no search loop.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, top_k_accuracy_score, f1_score
import xgboost as xgb

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
from google.colab import drive
print("Connecting to Google Drive...")
drive.mount('/content/drive')

print("Loading data...")
csv_path = "/content/drive/MyDrive/Final_Augmented_dataset_Diseases_and_Symptoms.csv"
df = pd.read_csv(csv_path)
print(f"Initial Dataset Shape: {df.shape}")

Connecting to Google Drive...
Mounted at /content/drive
Loading data...
Initial Dataset Shape: (246945, 378)


In [3]:
target_col = "diseases"
feature_cols = [c for c in df.columns if c != target_col]

# 1. Memory Optimization
X = df[feature_cols].astype(np.uint8)
y_raw = df[target_col]

# 2. Drop Zero-Variance Features
nunique = X.nunique()
useless_cols = nunique[nunique <= 1].index.tolist()
if useless_cols:
    X = X.drop(columns=useless_cols)

# 3. Drop Exact Duplicates
combined = X.copy()
combined['__target__'] = y_raw
dup_mask = combined.duplicated(keep='first').values
X = X[~dup_mask].reset_index(drop=True)
y_raw = y_raw[~dup_mask].reset_index(drop=True)

# 4. Drop Rare Classes (<2 samples)
class_counts = y_raw.value_counts()
rare_classes = class_counts[class_counts < 2].index
if len(rare_classes) > 0:
    mask = ~y_raw.isin(rare_classes)
    X = X[mask].reset_index(drop=True)
    y_raw = y_raw[mask].reset_index(drop=True)

# 5. Perfect Label Encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
new_num_classes = len(le.classes_)

print(f"Cleaned Data -> Rows: {X.shape[0]}, Features: {X.shape[1]}, Valid Classes: {new_num_classes}")

Cleaned Data -> Rows: 189601, Features: 328, Valid Classes: 727


In [4]:
# Notice we are passing y_encoded!
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=RANDOM_STATE, stratify=y_encoded
)

print(f"Training Data: {X_train.shape[0]} rows")
print(f"Testing Data:  {X_test.shape[0]} rows")

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

Training Data: 151680 rows
Testing Data:  37921 rows


## Single Tuned Training Run -- Leaf-Wise Growth
What changed from your last run, and why:
- `grow_policy="lossguide"` + `max_leaves=256` -- the core change. Trees grow toward the highest-value splits first instead of uniform depth expansion. `max_depth=0` means depth is now uncapped -- `max_leaves` is what actually bounds tree size instead.
- `min_child_weight`: 2 -> 4 -- leaf-wise growth can create very small, overfit leaves if left unchecked; raising this keeps leaves from getting too specific to a handful of rows.
- `reg_lambda`: 1.5 -> 2.2, added `max_delta_step=1` -- extra guardrails against the same overfitting risk, since leaf-wise growth is more aggressive by design.
- `eta`: 0.08 -> 0.06, rounds raised to 2000 with same `early_stopping_rounds=50` -- slightly slower learning rate for a more stable finish, offset by more available rounds; early stopping keeps actual runtime in check.

In [5]:
params = {
    "objective": "multi:softprob",
    "num_class": new_num_classes,
    "eval_metric": ["mlogloss", "merror"],
    "tree_method": "hist",
    "device": "cuda",
    "grow_policy": "lossguide",
    "max_depth": 0,          # uncapped -- max_leaves controls tree size instead
    "max_leaves": 256,
    "eta": 0.06,
    "gamma": 0.15,
    "subsample": 0.85,
    "colsample_bytree": 0.75,
    "min_child_weight": 4,
    "max_delta_step": 1,
    "reg_lambda": 2.2,
    "reg_alpha": 0.15,
    "seed": RANDOM_STATE,
}

print("Training Tuned XGBoost on GPU (leaf-wise growth, single pass)...")
evals_result = {}

xgb_model = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=[(dtrain, "train"), (dtest, "val")],
    early_stopping_rounds=50,
    evals_result=evals_result,
    verbose_eval=100,
)
print(f"\nXGBoost Best iteration: {xgb_model.best_iteration}")

Training Tuned XGBoost on GPU (leaf-wise growth, single pass)...
[0]	train-mlogloss:5.76919	train-merror:0.88159	val-mlogloss:5.76965	val-merror:0.88392
[100]	train-mlogloss:0.89134	train-merror:0.16362	val-mlogloss:0.99767	val-merror:0.21495
[200]	train-mlogloss:0.50595	train-merror:0.13293	val-mlogloss:0.64124	val-merror:0.20690
[227]	train-mlogloss:0.47993	train-merror:0.13016	val-mlogloss:0.62065	val-merror:0.20762

XGBoost Best iteration: 177


## Evaluate: Top-1 / Top-3 / Top-5 / Weighted F1 (vs. theoretical ceilings)

In [6]:
proba = xgb_model.predict(dtest, iteration_range=(0, xgb_model.best_iteration + 1))
pred = np.argmax(proba, axis=1)

top1 = accuracy_score(y_test, pred)
top3 = top_k_accuracy_score(y_test, proba, k=3, labels=np.arange(new_num_classes))
top5 = top_k_accuracy_score(y_test, proba, k=5, labels=np.arange(new_num_classes))
f1w = f1_score(y_test, pred, average="weighted", zero_division=0)

print("=== Leaf-Wise Tuned XGBoost Results ===")
print(f"Top-1 Accuracy:  {top1:.4f}   (theoretical ceiling: 0.8959)")
print(f"Top-3 Accuracy:  {top3:.4f}   (theoretical ceiling: 0.9821)")
print(f"Top-5 Accuracy:  {top5:.4f}   (theoretical ceiling: 0.9943)")
print(f"Weighted F1:     {f1w:.4f}")

xgb_model.save_model("xgb_tuned_lossguide.json")
print("\nSaved: xgb_tuned_lossguide.json")

=== Leaf-Wise Tuned XGBoost Results ===
Top-1 Accuracy:  0.7939   (theoretical ceiling: 0.8959)
Top-3 Accuracy:  0.9218   (theoretical ceiling: 0.9821)
Top-5 Accuracy:  0.9518   (theoretical ceiling: 0.9943)
Weighted F1:     0.7878

Saved: xgb_tuned_lossguide.json
